# Video Object Tracking on MOT16 — Midterm Demo**Group 4** · COMP-SCI 5567 · Faster R-CNN detection + Siamese re-identificationRuntime → **Change runtime type → T4 GPU** before running.This notebook runs the full pipeline end to end. The baseline uses theCOCO-pretrained detector, so it produces tracked video **without any training**.

## 1. Setup

In [ ]:
# Clone the repo and install it. Re-running this cell is safe.import os, pathlibREPO = "https://github.com/ben-blake/cs5567-project.git"if not pathlib.Path("/content/cs5567-project").exists():    !git clone -q {REPO} /content/cs5567-project%cd /content/cs5567-project!git pull -q!pip install -q -e . gdownprint("installed")

In [ ]:
# Mount Drive so checkpoints survive a runtime disconnect. Optional:# skip this and checkpoints land on local disk instead.from google.colab import drivedrive.mount('/content/drive')

In [ ]:
from mot.config import PATHS, describe_environmentprint(describe_environment())print()print("data       ", PATHS.data)print("checkpoints", PATHS.checkpoints)print("outputs    ", PATHS.outputs)PATHS.ensure()

## 2. DataMOT16 is ~1.9 GB and takes a few minutes. `--mot16-crops` needs no download:it cuts Re-ID training crops straight out of MOT16's ground truth, which arein-domain for exactly the footage we track.

In [ ]:
!python scripts/download_data.py --mot16

In [ ]:
from mot.data.mot16 import find_sequencesfrom mot.config import PATHStrain_sequences = find_sequences(PATHS.mot16 / "train")for sequence in train_sequences:    print(f"{sequence.name:12s} {sequence.info.seq_length:5d} frames  "          f"{sequence.info.im_width}x{sequence.info.im_height}  {sequence.info.frame_rate} fps")

### What one annotation row means

In [ ]:
from mot.data.mot16 import group_by_frame, group_by_tracksequence = train_sequences[0]ground_truth = sequence.ground_truth()print(f"{sequence.name}: {len(ground_truth)} pedestrian boxes, "      f"{len(group_by_track(ground_truth))} identities")first = ground_truth[0]print(f"\nframe={first.frame}  id={first.track_id}  "      f"box(xyxy)={tuple(round(v, 1) for v in first.box.as_xyxy())}  "      f"visibility={first.visibility}")

### A frame with its ground truth drawn on

In [ ]:
import numpy as npfrom PIL import Imageimport matplotlib.pyplot as pltfrom mot.viz.draw import draw_tracksframe_number = 1annotations = group_by_frame(ground_truth)[frame_number]image = Image.open(sequence.frame_path(frame_number))overlay = draw_tracks(    image,    np.array([a.box.as_xyxy() for a in annotations], dtype=np.float32),    [a.track_id for a in annotations],)plt.figure(figsize=(14, 8)); plt.imshow(overlay); plt.axis("off")plt.title(f"{sequence.name} frame {frame_number} — ground truth"); plt.show()

## 3. DetectionThe COCO-pretrained Faster R-CNN already detects people, so we can see realoutput before training anything.

In [ ]:
import torchfrom mot.detect.faster_rcnn import build_pretrained_person_detector, detect, build_model, parameter_summaryfrom mot.config import get_devicedevice = get_device()detector = build_pretrained_person_detector().eval().to(device)tensor = torch.from_numpy(np.array(image.convert("RGB"))).permute(2, 0, 1).float() / 255.0result = detect(detector, [tensor], score_threshold=0.7, device=device)[0]print(f"{len(result['boxes'])} people detected (threshold 0.7)")overlay = draw_tracks(image, result["boxes"].numpy(),                      list(range(1, len(result["boxes"]) + 1)),                      result["scores"].tolist())plt.figure(figsize=(14, 8)); plt.imshow(overlay); plt.axis("off")plt.title("Faster R-CNN detections (pretrained, no fine-tuning)"); plt.show()

In [ ]:
# Few-shot fine-tuning budget: how much of the model actually trains.summary = parameter_summary(build_model(pretrained=False, freeze_backbone=True))for name, count in summary.items():    print(f"{name:10s} {count:>12,}")print(f"trainable share: {100 * summary['trainable'] / summary['total']:.1f}%")

## 4. Re-identificationThe Siamese network maps a person crop to a 128-d unit vector. Same person →vectors close together.

In [ ]:
from mot.reid.siamese import SiameseNetwork, CROP_SIZE, cosine_similarity_matrix, embed_cropsfrom mot.pipeline import crop_peoplereid = SiameseNetwork().eval()print(f"crop size (HxW): {CROP_SIZE}")print(f"parameters: {sum(p.numel() for p in reid.parameters()):,}")crops = crop_people(image, result["boxes"].numpy())embeddings = embed_crops(reid, crops, device=device)print(f"embeddings: {tuple(embeddings.shape)}  (unit norm: {torch.allclose(embeddings.norm(dim=1), torch.ones(len(embeddings)), atol=1e-5)})")

In [ ]:
# NOTE: untrained weights -> this matrix is near-uniform. After training it# should be bright on the diagonal and dark off it. This is the "before" picture.similarity = cosine_similarity_matrix(embeddings, embeddings).numpy()plt.figure(figsize=(6, 5))plt.imshow(similarity, cmap="viridis"); plt.colorbar(label="cosine similarity")plt.title("Crop-to-crop similarity (untrained)"); plt.xlabel("crop"); plt.ylabel("crop"); plt.show()

### Training the Re-ID networkUses MOT16-derived crops. Expect pair accuracy to climb from ~0.5 (chance)toward 0.9+.

In [ ]:
!python scripts/download_data.py --mot16-crops --max-sequences 3

In [ ]:
from pathlib import Pathfrom torch.utils.data import DataLoaderfrom mot.data.datasets import ReIDPairDatasetfrom mot.data.augment import reid_transformfrom mot.reid.train import train_reid, pair_accuracy# Index the exported crops by identity, namespaced per sequence so the same# track number in two sequences is not treated as the same person.crop_root = PATHS.data / "mot16_reid_crops"identities = {}for sequence_dir in sorted(crop_root.iterdir()):    if not sequence_dir.is_dir():        continue    for path in sorted(sequence_dir.glob("*.jpg")):        key = f"{sequence_dir.name}_{path.name.split('_')[0]}"        identities.setdefault(key, []).append(path)identities = {i: paths for i, (_, paths) in enumerate(sorted(identities.items()))}print(f"{len(identities)} identities, {sum(len(v) for v in identities.values())} crops")train_loader = DataLoader(    ReIDPairDataset(identities, transform=reid_transform(True), pairs_per_epoch=4000),    batch_size=64, shuffle=True, num_workers=2,)val_loader = DataLoader(    ReIDPairDataset(identities, transform=reid_transform(False), pairs_per_epoch=800, seed=7),    batch_size=64, num_workers=2,)reid = SiameseNetwork()print("accuracy before training:", round(pair_accuracy(reid, val_loader, device=device), 3))

In [ ]:
history = train_reid(    reid, train_loader, epochs=8, device=device,    val_loader=val_loader,    checkpoint_path=PATHS.checkpoints / "reid_siamese.pt",    log_every=0,)

In [ ]:
epochs = [h["epoch"] for h in history]fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))ax1.plot(epochs, [h["loss"] for h in history], marker="o")ax1.set_xlabel("epoch"); ax1.set_ylabel("contrastive loss"); ax1.set_title("Training loss"); ax1.grid(alpha=.3)if "val_accuracy" in history[0]:    ax2.plot(epochs, [h["val_accuracy"] for h in history], marker="o", color="seagreen")    ax2.axhline(0.5, ls="--", c="gray", label="chance")    ax2.set_xlabel("epoch"); ax2.set_ylabel("pair accuracy"); ax2.set_title("Validation"); ax2.legend(); ax2.grid(alpha=.3)plt.tight_layout(); plt.savefig(PATHS.outputs / "reid_training.png", dpi=150); plt.show()

## 5. Tracking — the demoRuns detection + Re-ID + association over a sequence and writes a video.Keep `MAX_FRAMES` small for a live demo.

In [ ]:
from mot.pipeline import TrackingPipeline, render_results, longest_track, results_to_mot_formatfrom mot.track.tracker import Trackerimport timeSEQUENCE   = "MOT16-02"MAX_FRAMES = 150          # raise for the final videosequence = [s for s in train_sequences if s.name == SEQUENCE][0]pipeline = TrackingPipeline(    detector, reid=reid,    tracker=Tracker(max_age=30, min_hits=3, iou_threshold=0.3, appearance_weight=0.5),    score_threshold=0.7, device=device,)started = time.time()results = pipeline.run_sequence(sequence, max_frames=MAX_FRAMES)elapsed = time.time() - startedprint(f"\n{len(results)} frames in {elapsed:.1f}s ({len(results)/elapsed:.2f} FPS)")print(f"{sum(len(r) for r in results)} boxes, "      f"{len({i for r in results for i in r.track_ids})} unique identities")

In [ ]:
target = longest_track(results)frames_on_target = sum(target in r.track_ids for r in results)print(f"target id {target} appears in {frames_on_target}/{len(results)} frames "      f"({100*frames_on_target/len(results):.1f}%)")video_path = render_results(sequence, results, PATHS.outputs / f"{sequence.name}_demo.mp4",                            highlight_id=target)print("wrote", video_path)

In [ ]:
# Play it inline.from IPython.display import HTMLfrom base64 import b64encodedata = b64encode(open(video_path, "rb").read()).decode()HTML(f'<video width=900 controls><source src="data:video/mp4;base64,{data}" type="video/mp4"></video>')

## 6. EvaluationMOT16-02 is a train sequence, so it has ground truth to score against.

In [ ]:
from mot.eval.metrics import score_trackingfrom mot.data.mot16 import group_by_framegt_by_frame = group_by_frame(sequence.ground_truth())tracked_frames = {r.frame for r in results}gt_eval = {    f: (np.array([a.box.as_xyxy() for a in anns], dtype=np.float32), [a.track_id for a in anns])    for f, anns in gt_by_frame.items() if f in tracked_frames}pred_eval = {r.frame: (r.boxes, r.track_ids) for r in results}score = score_tracking(gt_eval, pred_eval)for name, value in score.as_dict().items():    print(f"{name:6s} {value:>9.3f}" if isinstance(value, float) else f"{name:6s} {value:>9}")

### Ablation: does Re-ID actually help?

In [ ]:
# Same detector, same thresholds, appearance disabled.pipeline_no_reid = TrackingPipeline(    detector, reid=None,    tracker=Tracker(max_age=30, min_hits=3, iou_threshold=0.3, appearance_weight=0.0),    score_threshold=0.7, device=device,)results_no_reid = pipeline_no_reid.run_sequence(sequence, max_frames=MAX_FRAMES)score_no_reid = score_tracking(gt_eval, {r.frame: (r.boxes, r.track_ids) for r in results_no_reid})print(f"{'metric':8s} {'with Re-ID':>12s} {'overlap only':>14s}")for key in ("MOTA", "MOTP", "IDSW"):    a, b = score.as_dict()[key], score_no_reid.as_dict()[key]    print(f"{key:8s} {a:>12.3f} {b:>14.3f}")

## 7. Save for the presentationCopies the video and figures to Drive so they survive the runtime.

In [ ]:
import shutilfrom mot.config import drive_mountedif drive_mounted():    destination = pathlib.Path("/content/drive/MyDrive/cs5567/presentation")    destination.mkdir(parents=True, exist_ok=True)    for item in PATHS.outputs.glob("*"):        if item.is_file():            shutil.copy2(item, destination / item.name)            print("saved", destination / item.name)else:    print("Drive not mounted; download from the Files panel instead.")